In [24]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

print("Libraries Imported Successfully!")

Libraries Imported Successfully!


In [25]:
weather = pd.read_csv("../../data/processed/weather/weather_processed.csv")

weather["datetime"] = pd.to_datetime(weather["datetime"])

print("Weather Dataset Loaded Successfully!")

Weather Dataset Loaded Successfully!


In [26]:
print(weather.shape)

weather.head()

(1629108, 7)


,datetime,city,temperature,humidity,pressure,wind_direction,weather_description
0,2012-10-01 12:00:00,Albuquerque,11.970000,50.0,1024.0,360.0,sky is clear
1,2012-10-01 13:00:00,Albuquerque,11.970000,50.0,1024.0,360.0,sky is clear
2,2012-10-01 14:00:00,Albuquerque,12.004558,49.0,1024.0,360.0,sky is clear
3,2012-10-01 15:00:00,Albuquerque,12.083952,49.0,1024.0,360.0,sky is clear
4,2012-10-01 16:00:00,Albuquerque,12.163345,49.0,1024.0,360.0,sky is clear


In [27]:
weather["temperature_category"] = pd.cut(
    weather["temperature"],
    bins=[-100,0,15,30,100],
    labels=[
        "Cold",
        "Cool",
        "Moderate",
        "Hot"
    ]
)

In [28]:
weather["temperature_category"].value_counts()

temperature_category
Moderate    808287
Cool        597713
Cold        130521
Hot          92587
Name: count, dtype: int64

In [29]:
weather["humidity_category"] = pd.cut(
    weather["humidity"],
    bins=[0,40,70,100],
    labels=[
        "Low",
        "Medium",
        "High"
    ]
)

In [30]:
weather["humidity_category"].value_counts()

humidity_category
High      833140
Medium    574499
Low       221469
Name: count, dtype: int64

In [31]:
weather["pressure_category"] = pd.cut(
    weather["pressure"],
    bins=[800,1005,1025,1100],
    labels=[
        "Low",
        "Normal",
        "High"
    ]
)

In [32]:
weather["pressure_category"].value_counts()

pressure_category
Normal    1141193
High       325405
Low        162508
Name: count, dtype: int64

In [33]:
def wind_direction(angle):

    if pd.isna(angle):
        return np.nan

    if angle >=315 or angle <45:
        return "North"

    elif angle <135:
        return "East"

    elif angle <225:
        return "South"

    else:
        return "West"


weather["wind_direction_category"] = weather["wind_direction"].apply(
    wind_direction
)

In [34]:
weather["wind_direction_category"].value_counts()

wind_direction_category
North    440421
West     437904
South    433620
East     317163
Name: count, dtype: int64

In [35]:
weather["rain_indicator"] = (
    weather["weather_description"]
    .str.contains("rain",case=False,na=False)
    .astype(int)
)

In [36]:
weather["snow_indicator"] = (
    weather["weather_description"]
    .str.contains("snow",case=False,na=False)
    .astype(int)
)

In [37]:
weather["storm_indicator"] = (
    weather["weather_description"]
    .str.contains("thunder",case=False,na=False)
    .astype(int)
)

In [38]:
weather["fog_indicator"] = (
    weather["weather_description"]
    .str.contains(
        "fog|mist|haze",
        case=False,
        na=False,
        regex=True
    )
    .astype(int)
)

In [39]:
def cloud_category(desc):

    if pd.isna(desc):
        return "Unknown"

    desc = desc.lower()

    if "clear" in desc:
        return "Clear"

    elif "few clouds" in desc:
        return "Partly Cloudy"

    elif "scattered" in desc:
        return "Cloudy"

    elif "broken" in desc:
        return "Mostly Cloudy"

    elif "overcast" in desc:
        return "Overcast"

    else:
        return "Other"


weather["cloud_category"] = weather["weather_description"].apply(
    cloud_category
)

In [40]:
def weather_severity(desc):

    if pd.isna(desc):
        return "Unknown"

    desc = desc.lower()

    high = [
        "thunder",
        "storm",
        "tornado",
        "heavy rain",
        "heavy snow"
    ]

    medium = [
        "rain",
        "snow",
        "fog",
        "mist",
        "haze",
        "drizzle"
    ]

    if any(x in desc for x in high):
        return "High"

    elif any(x in desc for x in medium):
        return "Medium"

    else:
        return "Low"


weather["weather_severity"] = weather["weather_description"].apply(
    weather_severity
)

In [41]:
weather["weather_severity"].value_counts()

weather_severity
Low       1250396
Medium     364053
High        14659
Name: count, dtype: int64

In [42]:
severity_score = {
    "Low":1,
    "Medium":2,
    "High":3,
    "Unknown":0
}

weather["weather_risk_score"] = (
    weather["weather_severity"]
    .map(severity_score)
)

In [43]:
weather["weather_risk_score"].describe()

count    1.629108e+06
mean     1.241464e+00
std      4.485038e-01
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      3.000000e+00
Name: weather_risk_score, dtype: float64

In [44]:
weather.info()

<class 'pandas.DataFrame'>
RangeIndex: 1629108 entries, 0 to 1629107
Data columns (total 18 columns):
 #   Column                   Non-Null Count    Dtype         
---  ------                   --------------    -----         
 0   datetime                 1629108 non-null  datetime64[us]
 1   city                     1629108 non-null  str           
 2   temperature              1629108 non-null  float64       
 3   humidity                 1629108 non-null  float64       
 4   pressure                 1629108 non-null  float64       
 5   wind_direction           1629108 non-null  float64       
 6   weather_description      1629108 non-null  str           
 7   temperature_category     1629108 non-null  category      
 8   humidity_category        1629108 non-null  category      
 9   pressure_category        1629106 non-null  category      
 10  wind_direction_category  1629108 non-null  str           
 11  rain_indicator           1629108 non-null  int64         
 12  snow_indica

In [45]:
weather.head()

,datetime,city,temperature,humidity,pressure,wind_direction,weather_description,temperature_category,humidity_category,pressure_category,wind_direction_category,rain_indicator,snow_indicator,storm_indicator,fog_indicator,cloud_category,weather_severity,weather_risk_score
0,2012-10-01 12:00:00,Albuquerque,11.970000,50.0,1024.0,360.0,sky is clear,Cool,Medium,Normal,North,0,0,0,0,Clear,Low,1
1,2012-10-01 13:00:00,Albuquerque,11.970000,50.0,1024.0,360.0,sky is clear,Cool,Medium,Normal,North,0,0,0,0,Clear,Low,1
2,2012-10-01 14:00:00,Albuquerque,12.004558,49.0,1024.0,360.0,sky is clear,Cool,Medium,Normal,North,0,0,0,0,Clear,Low,1
3,2012-10-01 15:00:00,Albuquerque,12.083952,49.0,1024.0,360.0,sky is clear,Cool,Medium,Normal,North,0,0,0,0,Clear,Low,1
4,2012-10-01 16:00:00,Albuquerque,12.163345,49.0,1024.0,360.0,sky is clear,Cool,Medium,Normal,North,0,0,0,0,Clear,Low,1


In [46]:
weather.to_csv(
    "../../data/processed/weather/weather_feature_engineered.csv",
    index=False
)

print("Weather Feature Engineered Dataset Saved Successfully!")

Weather Feature Engineered Dataset Saved Successfully!
